# Exoplanet Habitability

Goal: use the NASA Exoplanet Archive's Planetary Systems Composite Parameters table (`pscomppars`) to find rocky exoplanets in the habitable zone, then compare our list with NASA-linked research (Kopparapu et al. 2014, the HWO target stars, and Bohl et al. 2026).

## Setup

This cell finds the project folder and adds it to Python's search path, so we can import our own tested code with `from src import physics`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The notebook usually runs from inside notebooks/, so the project root is one level up
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import physics

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Create any missing folders (does nothing if they already exist)
for folder in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("pandas", pd.__version__, "| numpy", np.__version__)

### pandas 2 vs pandas 3: what changes for us

This notebook runs on pandas 2.x (what Google Colab has) and pandas 3.x (local). The setup cell above prints which version you are using. Two differences matter here:

**1. Copy-on-Write (the default in pandas 3).** A filtered DataFrame behaves like its own independent copy. So chained assignment such as
`df[df["pl_rade"] > 0]["label"] = "x"` silently does nothing in pandas 3 (pandas 2 only warns with `SettingWithCopyWarning`).
*Our rule:* always assign with `df.loc[mask, "col"] = value`, and call `.copy()` when we keep a filtered subset that we will modify later.

**2. New default string dtype.** Text columns such as `pl_name` show up as `str` in pandas 3 but `object` in pandas 2, so `df.dtypes` looks different in Colab. The data is the same.
*Our rule:* never select text columns with `select_dtypes(include="object")` (in pandas 3 it misses them). Use `pd.api.types.is_string_dtype(...)` or list the columns by name.

When a later output differs between versions, a note there points back to this cell.

## Phase 1: Load the data

### How we get the data: TAP and ADQL

**TAP** (Table Access Protocol) is a standard web interface that astronomy archives use to answer database queries. **ADQL** is the SQL-like language we send to it, for example `SELECT pl_name, pl_rade FROM pscomppars`. We send the query as a web request with `requests` and the archive sends back a CSV file.

### Why the Composite Parameters table (`pscomppars`)?

The default Planetary Systems table (`ps`) keeps **every published solution** for every planet, so one planet can appear many times. When we checked, `ps` had 40,350 rows, and TRAPPIST-1 e alone had 5 rows. You would have to pick one solution per planet yourself. `pscomppars` has **one row per planet**, filled with the best available value for each column, which is what a planet-by-planet analysis needs.

**Caveat from the archive:** the values in one row can come from different papers, so a row is *"not necessarily self-consistent"*. Some values are also **calculated** by the archive when a measurement is missing. For example, it estimates a radius from the mass using the Chen & Kipping (2017) relation. That's why we also keep three source columns: `pl_rade_reflink`, `pl_bmassprov` and `pl_insol_reflink`. ([column docs](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html), [calculated values](https://exoplanetarchive.ipac.caltech.edu/docs/pscp_calc.html))

In [ ]:
from datetime import date

from src import archive

COLUMNS = [
    "pl_name", "hostname", "discoverymethod", "disc_year",
    "pl_rade", "pl_bmasse", "pl_orbper", "pl_orbsmax", "pl_orbeccen",
    "pl_insol", "pl_eqt",
    "st_teff", "st_rad", "st_lum", "st_mass", "sy_dist",
    # Source columns: say whether a value was measured or calculated by the archive
    "pl_rade_reflink", "pl_bmassprov", "pl_insol_reflink",
]

adql = "SELECT " + ", ".join(COLUMNS) + " FROM pscomppars"
print(adql)

**Load or download.** If a snapshot already exists in `data/raw/`, we load the newest one and do **not** call the API. Set `REFRESH = True` to take a new dated snapshot; old snapshots are never overwritten.

In [ ]:
REFRESH = False  # set to True to take a new dated snapshot

todays_file = RAW_DIR / f"pscomppars_{date.today().isoformat()}.csv"
snapshot = archive.latest_snapshot(RAW_DIR, "pscomppars")

if snapshot is None or (REFRESH and not todays_file.exists()):
    print("Downloading a new snapshot from the archive...")
    snapshot = archive.download_tap_csv(adql, todays_file)
else:
    print("Loading existing snapshot (no API call)")

print("File:", snapshot.relative_to(PROJECT_ROOT).as_posix())
planets = pd.read_csv(snapshot)

# If the archive returned an error message instead of data, the header would not match
assert list(planets.columns) == COLUMNS, f"Unexpected columns: {list(planets.columns)}"

print("Shape (rows, columns):", planets.shape)
print(f"Planets in table: {len(planets):,}")

**Data types.** Numbers are `float64` or `int64`. Text columns show as `str` on pandas 3 and `object` on pandas 2; the data is the same (see the *pandas 2 vs pandas 3* cell).

In [ ]:
planets.dtypes.to_frame("dtype")

**Summary statistics.** `describe()` gives the count, mean, spread and percentiles of each numeric column. We transpose it (`.T`) so each column becomes a row, which is easier to read.

In [ ]:
planets.describe().T

**Percent missing per column.** For each column, the share of planets with no value. This tells us which later steps will lose planets.

In [ ]:
percent_missing = (
    planets.isna().mean()   # fraction of rows that are missing, per column
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .to_frame("percent_missing")
)
percent_missing

**Measured or calculated?** The reference column says "Calculated Value" when the archive computed the number itself instead of taking it from a paper. `pl_bmassprov` tells us whether the mass is a true mass, a minimum mass (`Msini`, from radial velocity), or estimated from the radius (`M-R relationship`).

In [ ]:
radius_calculated = planets["pl_rade_reflink"].str.contains("Calculated Value", na=False)
insol_calculated = planets["pl_insol_reflink"].str.contains("Calculated Value", na=False)

print(f"Radii calculated from mass by the archive: {radius_calculated.sum():,} "
      f"of {planets['pl_rade'].notna().sum():,} planets with a radius")
print(f"Insolations calculated by the archive:     {insol_calculated.sum():,} "
      f"of {planets['pl_insol'].notna().sum():,} planets with an insolation")
print()
print("Where does the mass come from? (pl_bmassprov)")
print(planets["pl_bmassprov"].value_counts(dropna=False).to_string())

In [ ]:
# Calculated radii by discovery method: radial velocity planets almost never have a measured radius
pd.crosstab(planets["discoverymethod"], radius_calculated.rename("radius_calculated"))

### What stands out (snapshot of 2026-10-24)

- **6,451 planets**, one row each. Radius is missing for only 0.8%, but eccentricity is missing for 16.6% and insolation for 9.2%. Dropping every row with *any* missing value would throw away far more planets than any single step needs (Phase 2).
- **Many radii are not measured.** 1,539 of 6,401 radii were calculated by the archive from the mass. That covers 1,134 of the radial velocity planets and all 292 microlensing planets, but only 2 transit planets. A "rocky" label based on these radii is really a mass-based guess.
- **Masses are often estimates too.** 3,035 masses come from the mass-radius relationship (mostly transit planets with no measured mass), and 924 are only minimum masses (`Msini`).
- **Odd values to handle in Phase 2.** 46 planets have `pl_insol` = 0. These are wide-orbit imaged planets whose tiny flux rounds to zero, not true zeros; we'll recompute S in Phase 3. 3,156 of 5,383 eccentricities are exactly 0, which usually means "assumed circular", not "measured circular". The hottest host (57,000 K) is a white dwarf in the NN Ser system.
- **Huge ranges** (orbital period from 0.09 days to 400 million days, insolation from 0 to 636,000 times Earth's). That's why we will use log scales for plots and clustering.

## Phase 2: Clean and filter

## Phase 3: Stellar energy

## Phase 4: Visualize

## Phase 5: Clustering

## Phase 6: Compare with NASA's own research